# FedXCrop: running the experiment grid

Runs the full experiment grid on a GPU runtime, with checkpoints and results on
Google Drive so a disconnected session resumes instead of restarting.

Before starting: **Runtime, Change runtime type, T4 GPU** (or better).

Run the cells in order. Sections 1 to 6 are setup and take about 15 minutes.
Section 7 is the grid itself and is the long part.


## 1. Check the GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime, Change runtime type, T4 GPU, then run this cell again.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 2. Mount Drive

Checkpoints and results live here, so nothing is lost when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/fedxcrop'
!mkdir -p {WORK}/runs {WORK}/results
print('working folder:', WORK)


## 3. Get the code

In [ ]:
%cd /content
![ -d fedxcrop_repo ] && (cd fedxcrop_repo && git pull -q) || git clone -q https://github.com/Mayan10/crp.git fedxcrop_repo
%cd /content/fedxcrop_repo
!git log --oneline -1


### Dependencies

Colab already has torch, torchvision, numpy, pandas, scikit-learn and
matplotlib. Only the missing packages are installed: installing
`requirements.txt` wholesale would downgrade Colab's CUDA build of torch.

The exact versions used are recorded in every run's `run_metadata.json`, so
whatever resolves here is captured with the results.

In [ ]:
!pip install -q captum imagehash

# Flower's simulation engine, which needs Ray. The pin is preferred for
# reproducibility; if it has no wheel for this runtime's Python, fall back to
# whatever does resolve rather than leaving the environment half installed.
!pip install -q 'flwr[simulation]==1.11.1' || pip install -q 'flwr[simulation]'

import flwr, ray, torch, torchvision
print('torch  ', torch.__version__)
print('flwr   ', flwr.__version__)
print('ray    ', ray.__version__)
print('\nRay imported, so the Flower simulation engine can run.')


## 4. Get the dataset

The split CSVs reference paths like `color/<class>/<file>.JPG`, so the folder
holding `color/` and `segmented/` is the dataset root. `grayscale/` is not used.

**Fastest route:** upload `plantvillage_colab.zip` to your Drive at
`MyDrive/fedxcrop/` first, then this cell just unzips it (about 2 minutes).

Otherwise it falls back to Kaggle, which needs `kaggle.json` uploaded to
`MyDrive/fedxcrop/`.

In [ ]:
import glob, os, shutil

DATASET = '/content/plantvillage dataset'

if not os.path.isdir(f'{DATASET}/color'):
    archives = glob.glob(f'{WORK}/*.zip')
    archives = [a for a in archives if 'kaggle' not in os.path.basename(a).lower()]
    if archives:
        print('unzipping', archives[0])
        !unzip -q '{archives[0]}' -d /content/
    else:
        print('no zip in Drive, falling back to Kaggle')
        !pip install -q kaggle
        os.makedirs('/root/.kaggle', exist_ok=True)
        if os.path.exists(f'{WORK}/kaggle.json'):
            shutil.copy(f'{WORK}/kaggle.json', '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
        !kaggle datasets download -d amgadalameri/plantvillage-dataset-zip -p /content --unzip

# The dataset may unzip one level deeper than expected. Find the real root.
if not os.path.isdir(f'{DATASET}/color'):
    found = glob.glob('/content/**/color', recursive=True)
    found = [f for f in found if os.path.isdir(f) and len(os.listdir(f)) > 30]
    if not found:
        raise SystemExit('Could not find a color/ folder with 38 classes under /content.')
    DATASET = os.path.dirname(found[0])
    print('dataset root detected as:', DATASET)

print('classes in color/    :', len(os.listdir(f'{DATASET}/color')))
print('classes in segmented/:', len(os.listdir(f'{DATASET}/segmented')))


### Link the dataset into the repository

Everything (scripts and tests alike) defaults to a `plantvillage dataset`
folder next to the code. Linking it there means no command below needs a
`data.root` override, and the test suite can find the images too.

In [ ]:
!rm -rf '/content/fedxcrop_repo/plantvillage dataset'
!ln -s '{DATASET}' '/content/fedxcrop_repo/plantvillage dataset'

# Confirm the images line up with the committed splits before spending GPU time.
import pandas as pd, pathlib
root = pathlib.Path('/content/fedxcrop_repo/plantvillage dataset')
test = pd.read_csv('/content/fedxcrop_repo/splits/test.csv')
missing = [p for p in test['path'].head(500) if not (root / p).is_file()]
print('missing of the first 500 test images:', len(missing))
if missing:
    raise SystemExit(f'Paths do not match, for example: {missing[0]}')
print('dataset matches the committed splits')


## 5. Put checkpoints and results on Drive

`runs/` (checkpoints, gitignored) is linked straight to Drive, so an
interrupted run resumes from its last completed round.

`results/` is synced rather than linked, so git stays happy with it. Run the
`sync()` helper after each group; it is also what makes `--skip-existing` know
which runs already finished after a restart.

In [ ]:
!rm -rf /content/fedxcrop_repo/runs
!ln -s {WORK}/runs /content/fedxcrop_repo/runs

# Bring back any results from a previous session.
!rsync -a {WORK}/results/ /content/fedxcrop_repo/results/

import os
CPUS = os.cpu_count()
print('CPUs available:', CPUS)
print('This pipeline is CPU bound on image augmentation, so the loader\n'
      'worker count matters. It is clamped to the CPU count automatically.')

def sync():
    """Copy results to Drive. Safe to call at any time, and often."""
    !rsync -a --exclude '*.pt' /content/fedxcrop_repo/results/ {WORK}/results/
    print('results synced to Drive')

sync()


## 6. Smoke test

Two short runs and the test suite. This is the gate: it costs a few minutes and
catches anything wrong with the setup before the grid does.

In [ ]:
!python scripts/run_federated.py --config configs/fedprox_noniid.yaml --smoke \
  --no-resume --engine sequential
!python scripts/run_federated.py --config configs/fedprox_noniid.yaml --smoke \
  --no-resume --engine flower


In [ ]:
!python -m pytest tests/ -q -m "not slow"


### The engine agreement test

The Flower engine could not be run on the machine this code was written on
(Ray has no build for that Python version), so this test is what establishes
that it does the same thing as the unit tested sequential reference.

**A skipped test here is a failed check.** The output must say `2 passed`, not
`2 skipped`.

In [ ]:
!python -m pytest tests/test_smoke.py -m slow -k "flower_engine_smoke or agree" -v -rs


If either test skipped, the dataset link or Ray is not set up: go back to
sections 3 to 5. If `test_both_engines_agree` **failed**, stop and report it
rather than running the grid, because the two engines are then not running the
same method.

## 7. Which side is the bottleneck

This pipeline can be limited by the GPU or by the loader, and which one
depends on the machine. The first grid run on a T4 spent over six minutes per
epoch because two CPU cores could not feed the GPU: the colour jitter alone is
about three quarters of the loader's cost.

`data.gpu_augment` moves the colour jitter onto the GPU. This measures both
settings on **this** machine, in about two minutes, so the choice is made from
a number rather than an assumption.


In [ ]:
!python scripts/benchmark_pipeline.py --batches 40


If it reports a speedup, leave the default alone (`gpu_augment` is already
true). If it reports roughly 1.0x or less, add `data.gpu_augment=false` to the
`--set` of every command below.

Either way the augmentation is the same: the two paths are checked against
each other in `tests/test_gpu_augment.py`.


## 7. Time one round

Measures on this runtime and projects the total, so you know what you are
committing to. This writes to a throwaway folder so it cannot be mistaken for a
real result later.

In [ ]:
import time

start = time.time()
!python -u scripts/run_federated.py --config configs/fedavg_iid.yaml --no-resume \
  --set federated.rounds=1 runs_dir=/content/timing_runs results_dir=/content/timing_results
per_round = time.time() - start

!rm -rf /content/timing_runs /content/timing_results

print(f'\none round, including startup: {per_round / 60:.1f} min')
print(f'one federated run (30 rounds): {30 * per_round / 3600:.1f} h')
print(f'24 federated runs:             {24 * 30 * per_round / 3600:.1f} h')
print(f'centralized (3 x 20 epochs):   {3 * 20 * per_round / 3600:.1f} h')
print(f'everything:                    {(24 * 30 + 60) * per_round / 3600:.1f} h')


Startup (imports, Ray, building the client shards) is a fixed cost of a minute
or two that this includes once per round, so the projection is on the
pessimistic side for a 30 round run.

If the total looks too large for the GPU time you have, stop here and say so:
the grid can be cut (fewer seeds, fewer rounds, fewer alphas) and it is better
to decide that now than half way through.

## 8. Run the grid

Every cell is restartable. Finished runs are skipped and an interrupted run
resumes from its last completed round, so if the session drops, re-run
sections 2 to 5 and then the cell you were on. Call `sync()` after each one.

### One decision first, if you already have centralized runs

Every configuration in the paper is compared against every other, so they all
need to have been trained through the same pipeline. Centralized runs finished
before the `gpu_augment` setting existed used the CPU path.

The next cell checks for that and tells you what to do. Re-running three
centralized seeds is a small fraction of the grid, and it removes a difference
between the baseline and everything it is compared against.


In [ ]:
import json, glob, os

# What pipeline did any already finished runs use?
stale = []
for meta_path in glob.glob('/content/fedxcrop_repo/runs/*/*/run_metadata.json'):
    config_path = os.path.join(os.path.dirname(meta_path), 'config.yaml')
    if not os.path.exists(config_path):
        continue
    import yaml
    saved = yaml.safe_load(open(config_path))
    used_gpu_augment = saved.get('data', {}).get('gpu_augment')
    if used_gpu_augment is None or used_gpu_augment is False:
        stale.append((os.path.basename(os.path.dirname(meta_path)), used_gpu_augment))

if stale:
    print('Runs trained with the CPU augmentation path:')
    for name, flag in stale:
        print(f'  {name}  (gpu_augment={flag})')
    print('\nIf the benchmark above favoured GPU augmentation, clear these and let\n'
          'them run again so the whole grid shares one pipeline:\n')
    print("  !rm -rf /content/fedxcrop_repo/runs/centralized")
    print("  !rm -rf /content/fedxcrop_repo/results/centralized {WORK}/results/centralized")
    print('\nIf it did not, add data.gpu_augment=false to every --set below instead,\n'
          'and keep the runs you already have.')
else:
    print('No finished runs, or all of them already match the current setting.')


In [ ]:
!python -u scripts/run_grid.py --group centralized
sync()


In [ ]:
!python -u scripts/run_grid.py --group fedavg_iid
sync()


In [ ]:
!python -u scripts/run_grid.py --group fedavg_noniid
sync()


### Choose mu on validation

The three candidate values are run at the hardest alpha, seed 0 only, and the
winner is chosen on validation accuracy. The test split takes no part in it.

In [ ]:
!python -u scripts/run_grid.py --group mu_selection
!python scripts/select_mu.py
sync()


**Set `SELECTED_MU` below to the value `select_mu.py` just reported**, then run
the rest. It is 0.01 only as a placeholder.

In [ ]:
SELECTED_MU = 0.01   # <-- set this from the cell above

!python -u scripts/run_grid.py --group fedprox_noniid --mu {SELECTED_MU}
sync()


In [ ]:
# Reproduces the original protocol for comparison. Needs centralized seed 0.
!python -u scripts/run_grid.py --group legacy
sync()


## 9. Explainability

Attribution metrics over the fixed 380 image sample, the same images for every
model. The first model listed is the reference the others are compared
against.

In [ ]:
!python -u scripts/run_xai.py --sanity --models \
  centralized_seed0 \
  fedavg_dirichlet_alpha0.1_K5_seed0 fedprox_dirichlet_alpha0.1_K5_mu{SELECTED_MU}_seed0 \
  fedavg_dirichlet_alpha0.5_K5_seed0 fedprox_dirichlet_alpha0.5_K5_mu{SELECTED_MU}_seed0
sync()


## 10. Tables, figures, and the report

In [ ]:
!python scripts/make_figures.py
!python scripts/check_pseudo_masks.py
!python scripts/make_xai_figures.py --models \
  centralized_seed0 fedprox_dirichlet_alpha0.1_K5_mu{SELECTED_MU}_seed0
!python scripts/make_report.py
sync()


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('/content/fedxcrop_repo/RESULTS.md').read()))


## 11. Bring the results home

Everything needed is now in `MyDrive/fedxcrop/results`. Download that folder,
copy it into `results/` in your local clone, and commit from there so the
commits carry your identity.

`runs/` holds checkpoints and is deliberately not committed.

In [ ]:
!cp /content/fedxcrop_repo/RESULTS.md {WORK}/
sync()
!du -sh {WORK}/results
!find {WORK}/results -name '*.csv' -o -name '*.json' -o -name '*.png' | wc -l
print('\nDownload MyDrive/fedxcrop/results and RESULTS.md, then commit locally.')
